# START

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

In [ ]:

root_path = "../data"

In [3]:
rawdata_path = os.path.join(root_path,'01-raw')
outcome_path = os.path.join(root_path,'02-input','outcomes')
tr_actual_path = os.path.join(root_path,'06-treatment_response','measured')

# Read in ROI Metadata

In [4]:
aoi_metadata_full = pd.read_excel(os.path.join(rawdata_path,'metadata',"meta_data.xlsx"), index_col=0)
aoi_metadata_full = aoi_metadata_full.set_index('DCC_ID')

ID is currently the AOI dcc id but we want to create an ROI_ID that refers to the LEAP ID and the ROI ID (using TISSUE_ID instead of LEAP_ID to avoid duplicates where a tissue was scanned more than once)

(there will be 2 AOI ids for each ROI - CD45 and PanCK - so we need to combine them)

In [5]:
aoi_metadata_full = aoi_metadata_full[aoi_metadata_full['Sample_Type'] != 'PELLET']
aoi_metadata_full['ID'] = aoi_metadata_full['Tissue_ID'] +"_"+ aoi_metadata_full['Roi'].astype(str)

In [ ]:
aoi_metadata = aoi_metadata_full[['ID','Tissue_ID','LEAP_ID','Roi','Segment', 'ROICoordinateX',	'ROICoordinateY','Response','Sample_Type','Scan.Name']]

In [8]:
aoi_metadata['Segment'].value_counts()

Segment
PanCK+          2374
CD45+           2102
PanCK-/CD45-     412
PanCK-            55
TLS                3
Full ROI           2
Name: count, dtype: int64

Get list of AOIs for each segment - this is the dcc id for the AOI (NOT the associated ROI number)

In [10]:
aois_cd45 = list(aoi_metadata[aoi_metadata['Segment']=='CD45+'].index)
aois_panck = list(aoi_metadata[aoi_metadata['Segment']=='PanCK+'].index)
print(f"{len(aois_cd45)} CD45+ AOIs | {len(aois_panck)} PanCK+ AOIs\ntotal {len(aois_cd45)+len(aois_panck)}")

2102 CD45+ AOIs | 2374 PanCK+ AOIs
total 4476


expression and metadata is currently at aoi level

## Keep only CD45 and PanCK AOIs

In [11]:
aoi_metadata = aoi_metadata.loc[aois_panck+aois_cd45]
aoi_metadata.shape

(4476, 10)

In [12]:
aoi_metadata.head()

,ID,Scan.Name,Tissue_ID,LEAP_ID,Roi,Segment,ROICoordinateX,ROICoordinateY,Response,Sample_Type
DCC_ID,,,,,,,,,,
DSP-1001660008694-G-A03.dcc,LEAP015_A_1.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,1.0,PanCK+,76238.40625,34222.16797,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A05.dcc,LEAP015_A_2.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,2.0,PanCK+,75587.60938,31947.06250,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A07.dcc,LEAP015_A_3.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,3.0,PanCK+,76252.50000,29973.66797,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A09.dcc,LEAP015_A_4.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,4.0,PanCK+,75719.64063,27396.35938,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A11.dcc,LEAP015_A_5.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,5.0,PanCK+,76045.59375,20348.43555,Non-Responder,DIAGNOSTIC


In [20]:
aoi_metadata.sort_values(['LEAP_ID','Roi']).head()

,ID,Scan.Name,Tissue_ID,LEAP_ID,Roi,Segment,ROICoordinateX,ROICoordinateY,Response,Sample_Type
DCC_ID,,,,,,,,,,
DSP-1001660020816-A-A03.dcc,LEAP001_1.0,Repeat: LEAP001_LEAP003_LEAP044,LEAP001,LEAP001,1.0,PanCK+,72508.83594,17484.98438,Non-Responder,DIAGNOSTIC
DSP-1001660020816-A-A05.dcc,LEAP001_2.0,Repeat: LEAP001_LEAP003_LEAP044,LEAP001,LEAP001,2.0,PanCK+,72508.83594,18678.61328,Non-Responder,DIAGNOSTIC
DSP-1001660020816-A-A04.dcc,LEAP001_2.0,Repeat: LEAP001_LEAP003_LEAP044,LEAP001,LEAP001,2.0,CD45+,72508.83594,18678.61328,Non-Responder,DIAGNOSTIC
DSP-1001660020816-A-A06.dcc,LEAP001_3.0,Repeat: LEAP001_LEAP003_LEAP044,LEAP001,LEAP001,3.0,CD45+,73699.89844,17484.98438,Non-Responder,DIAGNOSTIC
DSP-1001660020816-A-A09.dcc,LEAP001_4.0,Repeat: LEAP001_LEAP003_LEAP044,LEAP001,LEAP001,4.0,PanCK+,73699.89844,18678.61328,Non-Responder,DIAGNOSTIC


In [19]:
aoi_metadata[aoi_metadata['LEAP_ID']=='LEAP015'].sort_values('Roi').head()

,ID,Scan.Name,Tissue_ID,LEAP_ID,Roi,Segment,ROICoordinateX,ROICoordinateY,Response,Sample_Type
DCC_ID,,,,,,,,,,
DSP-1001660008694-G-A03.dcc,LEAP015_A_1.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,1.0,PanCK+,76238.40625,34222.16797,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A02.dcc,LEAP015_A_1.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,1.0,CD45+,76238.40625,34222.16797,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A05.dcc,LEAP015_A_2.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,2.0,PanCK+,75587.60938,31947.06250,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A07.dcc,LEAP015_A_3.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,3.0,PanCK+,76252.50000,29973.66797,Non-Responder,DIAGNOSTIC
DSP-1001660008694-G-A06.dcc,LEAP015_A_3.0,"LEAP015, LEAP016",LEAP015_A,LEAP015,3.0,CD45+,76252.50000,29973.66797,Non-Responder,DIAGNOSTIC


In [13]:
# make sure our ID is now unique within segments
grp = aoi_metadata.groupby(['ID','Segment'])[['Roi','Segment']].count()
assert(len(grp['Roi'].unique())==1)

In [21]:
aoi_metadata.Segment.value_counts()

Segment
PanCK+    2374
CD45+     2102
Name: count, dtype: int64

In [22]:
aoi_metadata.Sample_Type.value_counts()

Sample_Type
DIAGNOSTIC    3617
RESECTION      715
Pre-NACT        77
1_1             51
2_2             16
Name: count, dtype: int64

# Read in ROI Expression data

In [23]:
raw_matrix = pd.read_csv(os.path.join(rawdata_path,'sge',"raw_counts.csv"), 
                 sep=";",               # Semicolon separator
                 decimal=",",          # Comma as decimal point
                 quotechar='"',        # Handle quoted strings
                 index_col=0)          # Use the first column as row index (gene name)

expression_matrix = raw_matrix.astype(float)
# Replace '.' with '-' in column names
expression_matrix.columns = expression_matrix.columns.str.replace('.', '-', regex=False)
expression_matrix.columns = expression_matrix.columns.str.replace('-dcc', '.dcc', regex=False)
expression_matrix.head()

,DSP-1001660008694-G-A02.dcc,DSP-1001660008694-G-A03.dcc,DSP-1001660008694-G-A05.dcc,DSP-1001660008694-G-A06.dcc,DSP-1001660008694-G-A07.dcc,DSP-1001660008694-G-A08.dcc,DSP-1001660008694-G-A09.dcc,DSP-1001660008694-G-A10.dcc,DSP-1001660008694-G-A11.dcc,DSP-1001660008694-G-A12.dcc,...,DSP-1001660024022-D-H03.dcc,DSP-1001660024022-D-H04.dcc,DSP-1001660024022-D-H05.dcc,DSP-1001660024022-D-H06.dcc,DSP-1001660024022-D-H07.dcc,DSP-1001660024022-D-H08.dcc,DSP-1001660024022-D-H09.dcc,DSP-1001660024022-D-H10.dcc,DSP-1001660024022-D-H11.dcc,DSP-1001660024022-D-H12.dcc
A2M,5.0,11.0,12.0,2.0,26.0,12.0,25.0,1.0,53.0,7.0,...,21.0,12.0,314.0,20.0,7.0,52.0,134.0,19.0,59.0,458.0
ACADM,1.0,4.0,12.0,1.0,12.0,1.0,11.0,2.0,10.0,2.0,...,7.0,2.0,39.0,7.0,2.0,35.0,51.0,23.0,26.0,67.0
ACADS,1.0,2.0,3.0,1.0,6.0,1.0,8.0,1.0,6.0,2.0,...,4.0,7.0,28.0,5.0,4.0,20.0,22.0,5.0,5.0,44.0
ACAT1,1.0,9.0,7.0,1.0,45.0,1.0,11.0,1.0,9.0,4.0,...,7.0,9.0,55.0,2.0,5.0,35.0,57.0,15.0,34.0,47.0
ACVRL1,1.0,1.0,4.0,1.0,7.0,2.0,3.0,1.0,2.0,2.0,...,6.0,1.0,11.0,8.0,1.0,21.0,29.0,7.0,14.0,42.0


## Apply preprocessing required

In [24]:
log_transformed_expr = expression_matrix.apply(np.log1p).T

# Construct X and y

need to get the complete list of genes that we infer

In [25]:
genes_CD45 = joblib.load(os.path.join(root_path,'06-treatment_response','cd45_genelist_full.joblib'))
genes_PanCK = joblib.load(os.path.join(root_path,'06-treatment_response','panck_genelist_full.joblib'))
gene_count = len(genes_CD45)+len(genes_PanCK)
print(f"{len(genes_CD45)} CD45 genes | {len(genes_PanCK)} PanCK genes | {gene_count} total genes")

104 CD45 genes | 115 PanCK genes | 219 total genes


In [26]:
expr_cd45 = log_transformed_expr.loc[aois_cd45,genes_CD45]
expr_panck = log_transformed_expr.loc[aois_panck,genes_PanCK]

In [27]:
expr_cd45.head(1)

,AARD,ACTG2,ADAM15,APOC1,APOD,APOE,AREG,BCL2,BOD1L2,C1QL2,...,SUPT7L,TAPBP,TARP,TENT5C,TMEM106C,TPSAB1,TRPM3,UBE2E1,USP17L3,ZFP57
DSP-1001660008694-G-A02.dcc,0.693147,0.693147,1.791759,0.693147,0.693147,0.693147,0.693147,1.098612,0.693147,0.693147,...,1.791759,1.386294,0.693147,0.693147,2.079442,1.609438,0.693147,0.693147,1.098612,0.693147


ID is currently the AOI dcc id but we want to be able to merge AOIs by ROI ID so use the iD we created in metadata 

In [ ]:
#maintain a mapping of ID to ROI_ID

In [28]:
expr_cd45['ID'] =expr_cd45.index.map(aoi_metadata['ID'])
expr_cd45 = expr_cd45.reset_index().set_index('ID').rename(columns={"index": "ROI_ID"})

expr_panck['ID'] =expr_panck.index.map(aoi_metadata['ID'])
expr_panck = expr_panck.reset_index().set_index('ID').rename(columns={"index": "ROI_ID"})


merge the expression dfs (on ROI ID) so there is one row per ROI that has expression for both CD45 and PanCK AOI expression

In [29]:
expr_cd45 = expr_cd45.add_suffix("_CD45")
expr_panck = expr_panck.add_suffix("_PanCK")
expr_actual = expr_cd45.join(expr_panck, how='left',rsuffix='_EXTRA')
# drop the old AOI IDs
expr_actual = expr_actual.drop(columns=expr_actual.filter(like='ROI').columns)
expr_actual.shape

(2102, 219)

In [30]:
expr_actual.head(1)

,AARD_CD45,ACTG2_CD45,ADAM15_CD45,APOC1_CD45,APOD_CD45,APOE_CD45,AREG_CD45,BCL2_CD45,BOD1L2_CD45,C1QL2_CD45,...,TAF6_PanCK,TFF3_PanCK,TMA7_PanCK,TMEM106C_PanCK,TP53_PanCK,TP63_PanCK,TPI1_PanCK,UBD_PanCK,VIM_PanCK,ZEB1_PanCK
ID,,,,,,,,,,,,,,,,,,,,,
LEAP015_A_1.0,0.693147,0.693147,1.791759,0.693147,0.693147,0.693147,0.693147,1.098612,0.693147,0.693147,...,0.693147,0.693147,3.401197,2.833213,0.693147,0.693147,1.791759,0.693147,3.526361,0.693147


check we have one column per gene and no extras left from the merge

In [31]:
assert(expr_actual.shape[1]==gene_count)

For treatment response we want to keep only ROIs that have both cd45 and panck AOIs
so we have to drop any ROIs where we have NaNs from the merge

In [32]:
expr_actual[expr_actual.isna().any(axis=1)]

,AARD_CD45,ACTG2_CD45,ADAM15_CD45,APOC1_CD45,APOD_CD45,APOE_CD45,AREG_CD45,BCL2_CD45,BOD1L2_CD45,C1QL2_CD45,...,TAF6_PanCK,TFF3_PanCK,TMA7_PanCK,TMEM106C_PanCK,TP53_PanCK,TP63_PanCK,TPI1_PanCK,UBD_PanCK,VIM_PanCK,ZEB1_PanCK
ID,,,,,,,,,,,,,,,,,,,,,
LEAP005_10.0,1.098612,1.791759,4.499810,1.609438,1.945910,2.772589,2.708050,1.386294,0.693147,1.098612,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LEAP005_12.0,0.693147,0.693147,3.178054,2.484907,1.609438,2.772589,0.693147,0.693147,1.098612,0.693147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LEAP034_39.0,0.693147,0.693147,1.098612,0.693147,0.693147,0.693147,0.693147,1.098612,0.693147,0.693147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LEAP004_8.0,3.135494,3.871201,6.226537,5.762051,3.610918,6.822197,3.044522,4.369448,3.784190,4.795791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LEAP004_11.0,2.833213,3.688879,5.717028,4.276666,3.401197,4.624973,2.995732,3.761200,3.465736,4.276666,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LEAP145_31.0,1.945910,2.197225,4.025352,4.234107,2.197225,5.117994,1.791759,3.465736,1.791759,1.386294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LEAP145_37.0,2.079442,3.044522,5.231109,4.584967,2.397895,5.786897,2.708050,3.951244,3.218876,2.564949,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LEAP145_38.0,1.386294,1.791759,4.418841,3.637586,1.791759,4.532599,1.945910,3.295837,2.397895,0.693147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
expr_actual = expr_actual.dropna()

Group metadata to ROI level from AOI

In [39]:
roi_metadata = aoi_metadata.groupby('ID').max()
roi_metadata = roi_metadata.drop(columns=['Segment','Scan.Name','Tissue_ID'])
roi_metadata.shape

(2539, 6)

In [40]:
roi_metadata = roi_metadata.loc[expr_actual.index]
assert(expr_actual.shape[0] == roi_metadata.shape[0])
roi_metadata.shape

(1937, 6)

Save full

In [41]:
roi_metadata['Sample_Type'].value_counts()

Sample_Type
DIAGNOSTIC    1619
RESECTION      269
Pre-NACT        31
1_1             18
Name: count, dtype: int64

In [42]:
roi_metadata.to_csv(os.path.join(tr_actual_path,'roi_metadata_bothsegments_allsampletypes.csv'))
expr_actual.to_csv(os.path.join(tr_actual_path,'expr_bothsegments_allsampletypes.csv'))

# Restrict to cores only

why are we going back to aoi_metadata instead of using roi_metadata?

In [ ]:
# rois_metadata_cores = aoi_metadata[aoi_metadata['Sample_Type']=='DIAGNOSTIC']
# rois_metadata_cores.shape
# rois_metadata_cores = rois_metadata_cores[rois_metadata_cores['Response'].isin(['Responder','Non-Responder'])]
# aois_cd45_cores = list(rois_metadata_cores[rois_metadata_cores['Segment']=='CD45+'].index)
# aois_panck_cores = list(rois_metadata_cores[rois_metadata_cores['Segment']=='PanCK+'].index)
# expr_cd45_cores = log_transformed_expr.loc[aois_cd45_cores,genes_CD45]
# expr_panck_cores = log_transformed_expr.loc[aois_panck_cores,genes_PanCK]
# expr_cd45_cores['ID'] =expr_cd45_cores.index.map(rois_metadata_cores['ID'])
# expr_cd45_cores = expr_cd45_cores.reset_index().set_index('ID').rename(columns={"index": "ROI_ID"})

# expr_panck_cores['ID'] =expr_panck_cores.index.map(rois_metadata_cores['ID'])
# expr_panck_cores = expr_panck_cores.reset_index().set_index('ID').rename(columns={"index": "ROI_ID"})

# rois_metadata_cores = rois_metadata_cores.groupby('ID').max()
# expr_cd45_cores = expr_cd45_cores.add_suffix("_CD45")
# expr_panck_cores = expr_panck_cores.add_suffix("_PanCK")
# expr_cores = expr_cd45_cores.join(expr_panck_cores, how='left',rsuffix='_EXTRA')
# expr_cores.shape

# expr_cores = expr_cores.drop(columns=expr_cores.filter(like='ROI').columns)
# expr_cores = expr_cores.dropna()
# expr_cores.shape

# rois_metadata_cores = rois_metadata_cores.loc[expr_cores.index]
# assert(expr_cores.shape[0] == rois_metadata_cores.shape[0])
# rois_metadata_cores.shape

In [101]:
rois_metadata_cores = roi_metadata[roi_metadata['Sample_Type']=='DIAGNOSTIC']
rois_metadata_cores.head()

,LEAP_ID,Roi,ROICoordinateX,ROICoordinateY,Response,Sample_Type
ID,,,,,,
LEAP015_A_1.0,LEAP015,1.0,76238.40625,34222.16797,Non-Responder,DIAGNOSTIC
LEAP015_A_3.0,LEAP015,3.0,76252.50000,29973.66797,Non-Responder,DIAGNOSTIC
LEAP015_A_4.0,LEAP015,4.0,75719.64063,27396.35938,Non-Responder,DIAGNOSTIC
LEAP015_A_5.0,LEAP015,5.0,76045.59375,20348.43555,Non-Responder,DIAGNOSTIC
LEAP015_A_6.0,LEAP015,6.0,73764.54688,19207.20898,Non-Responder,DIAGNOSTIC


need to remove anything that is not Responder or Non-Responder

In [102]:
rois_metadata_cores['Response'].value_counts()

Response
Responder              1020
Non-Responder           570
NACT For Recurrence      20
Had Adjuvant Chemo        9
Name: count, dtype: int64

In [110]:
rois_metadata_cores = rois_metadata_cores[rois_metadata_cores['Response'].isin(['Responder','Non-Responder'])]
rois_metadata_cores.shape

(1590, 6)

In [114]:
expr_cores = expr_actual.loc[rois_metadata_cores.index]
assert(rois_metadata_cores.shape[0]==expr_cores.shape[0])
assert(~expr_cores.isna().any().any())

In [116]:
rois_metadata_cores.loc[expr_cores.index]['Sample_Type'].value_counts()

Sample_Type
DIAGNOSTIC    1590
Name: count, dtype: int64

In [117]:
rois_metadata_cores = rois_metadata_cores.sort_index()
expr_cores = expr_cores.sort_index()

In [118]:
rois_metadata_cores.to_csv(os.path.join(tr_actual_path,'roi_meta_cores_nodup.csv'))
expr_cores.to_csv(os.path.join(tr_actual_path,'expr_cores_nodup.csv'))

# END